In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
print(documents[0])

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [16]:
response = openai_client.beta.chat.completions.parse(
    model="openai/gpt-5",
    messages=messages,
    response_format=Questions
)

In [22]:
result = response.choices[0].message.parsed

print(result)

questions=['Is it too late to enroll now, or can I still hop in?', 'If I join late, can I still get a certificate?', 'What do I need to do to qualify for the certificate if I start now?', 'Can I just follow along now, and the only deadline is for the certificate?', 'Any deadlines I should worry about if I enroll today?']


In [23]:
print(result.questions)

['Is it too late to enroll now, or can I still hop in?', 'If I join late, can I still get a certificate?', 'What do I need to do to qualify for the certificate if I start now?', 'Can I just follow along now, and the only deadline is for the certificate?', 'Any deadlines I should worry about if I enroll today?']


In [34]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

from evaluation_utils import llm_structured

In [35]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to sign up now, or can I still join?', 'If I enroll now, can I still earn the certificate?', 'What are the conditions to get the certificate when joining mid-course?', 'Do I need to turn in a project to be eligible for the certificate?', 'If submissions are already closed, can I still take the course (just without the certificate)?']


In [36]:
usage.input_tokens, usage.output_tokens

AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'

In [39]:
print(usage)

CompletionUsage(completion_tokens=1439, prompt_tokens=210, total_tokens=1649, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=1344, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 10, 'engine_ttft_ms': 60, 'engine_ttlt_ms': 13838, 'pre_inference_ms': 148, 'service_tbt_ms': 10, 'service_ttft_ms': 473, 'service_ttlt_ms': 14247, 'total_duration_ms': 14107, 'user_visible_ttft_ms': 324})


In [41]:
print("Prompt tokens     :", usage.prompt_tokens)
print("Completion tokens :", usage.completion_tokens)
print("Total tokens      :", usage.total_tokens)

Prompt tokens     : 210
Completion tokens : 1439
Total tokens      : 1649


In [42]:
# names changes with different APIs:
# Responses API	   |   Chat Completions API
# input_tokens	   |   prompt_tokens
# output_tokens	   |   completion_tokens
# total_tokens	   |   total_tokens

In [45]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)
from evaluation_utils import calc_price

In [46]:
cost = calc_price(usage)

cost

{'input_cost': 0.0001575,
 'output_cost': 0.0064754999999999995,
 'total_cost': 0.006632999999999999}

In [47]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to sign up now, or can I still join?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll now, can I still earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What are the conditions to get the certificate when joining mid-course?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to turn in a project to be eligible for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'If submissions are already closed, can I still take the course (just without the certificate)?',
  'document': '74eb249bbf'}]

In [48]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)
from evaluation_utils import llm_structured_retry

In [49]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [50]:
generate_ground_truth(doc)

([{'question': 'I just discovered llm-zoomcamp—can I still jump in at this point?',
   'document': '74eb249bbf'},
  {'question': 'If I enroll late, can I still get a certificate, and what exactly do I need to do for that?',
   'document': '74eb249bbf'},
  {'question': 'Is there any requirement tied to starting now, like having to turn in the project before the submission window closes to be eligible for a certificate?',
   'document': '74eb249bbf'},
  {'question': 'I missed the start date; can I join now and still qualify for a certificate if I submit my project while submissions are open?',
   'document': '74eb249bbf'},
  {'question': 'Can I begin mid-course, and what’s the deadline-related condition for receiving the certificate?',
   'document': '74eb249bbf'}],
 CompletionUsage(completion_tokens=1095, prompt_tokens=210, total_tokens=1305, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=960, rejected_prediction_tokens=0

In [51]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [52]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [ ]:
# didn't able to run above few cells cause of API limit